In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here


import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# /kaggle/input/q1-stage-3-2026/PlantVillage
# Paths
train_dir = os.path.join(path,"PlantVillage", "train")
test_dir  = os.path.join(path,"PlantVillage", "test")

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# Datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Show samples
classes = train_dataset.classes

images, labels = next(iter(train_loader))

plt.figure(figsize=(8,4))

for i in range(6):
    plt.subplot(2,3,i+1)
    img = images[i].permute(1,2,0)*0.5 + 0.5
    plt.imshow(img)
    plt.title(classes[labels[i]])
    plt.axis("off")

plt.show()


In [ ]:
# Write your code here
import torch.nn as nn
import torch.nn.functional as F

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )

        self.conv5 = nn.Sequential(
            nn.Conv2d(256,512,3,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*1*1,256),
            nn.ReLU(),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):
        x = self.pool(self.conv1(x)) # 16
        x = self.pool(self.conv2(x)) # 8 نصهااا وكذا
        x = self.pool(self.conv3(x)) # 4
        x = self.pool(self.conv4(x)) # 22
        x = self.pool(self.conv5(x)) # 1

        x = self.fc(x)

        return x


In [ ]:
# Write your code here
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs,1)

        correct += (preds==labels).sum().item()
        total += labels.size(0)

    acc = correct/total
    return total_loss/len(loader), acc

def validate(model, loader, criterion, device):

    model.eval() # فاليد
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad(): # نوو

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs,1)

            correct += (preds==labels).sum().item()
            total += labels.size(0)

    acc = correct/total

    return total_loss/len(loader), acc


In [ ]:
# Write your code here
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5

train_losses=[]
val_losses=[]
train_acc=[]
val_acc=[]

for epoch in range(epochs):
    tl, ta = train_one_epoch(model,train_loader,optimizer,criterion,device)
    vl, va = validate(model,test_loader,criterion,device)

    train_losses.append(tl)
    val_losses.append(vl)

    train_acc.append(ta)
    val_acc.append(va)

    print(f"Epoch {epoch+1}: Train Acc={ta:.3f} Val Acc={va:.3f}")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), train_acc, label="Train Accuracy", marker='o')
plt.plot(range(1, epochs+1), val_acc, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()


In [ ]:
class ResidualCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(128,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(64,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        self.pool = nn.MaxPool2d(2)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,num_classes)
        )
    def forward(self,x):

        x1 = self.pool(self.conv1(x))   # 16
        x2 = self.pool(self.conv2(x1)) # 8

        x3 = self.pool(self.conv3(x2)) # 4
        x4 = self.pool(self.conv4(x3)) # 2

        # Fix residual size
        res = self.pool(self.pool(x2)) #
        # Skip connection (sum)
        x4 = x4 + res

        x5 = self.pool(self.conv5(x4)) # 1x1

        out = self.fc(x5)
        return out


In [ ]:
# Write your code here
# ريترين
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResidualCNN().to(device)

criterion = nn.CrossEntropyLoss() # مدري هل يفرق؟
# برججع هنا

optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3 #

train_losses=[]
val_losses=[]
train_acc=[]
val_acc=[]

for epoch in range(epochs):

    tl, ta = train_one_epoch(model,train_loader,optimizer,criterion,device)

    vl, va = validate(model,test_loader,criterion,device)

    train_losses.append(tl)
    val_losses.append(vl)

    train_acc.append(ta)
    val_acc.append(va)

    print(f"Epoch {epoch+1}: Train Acc={ta:.3f} Val Acc={va:.3f}")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), train_acc, label="Train Accuracy", marker='o')
plt.plot(range(1, epochs+1), val_acc, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()
